In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/content")

print("Available CSV files:")
for file in sorted(DATA_DIR.glob("*.csv")):
    print(file.name)

Available CSV files:
cbc_cleaned.csv
cmp_cleaned.csv


In [6]:
# Check the two cleaned datasets
cbc_cleaned = pd.read_csv(DATA_DIR / "cbc_cleaned.csv")
cmp_cleaned = pd.read_csv(DATA_DIR / "cmp_cleaned.csv")
print("CBC shape:", cbc_cleaned.shape)
print("CMP shape:", cmp_cleaned.shape)

print("\nCBC columns:")
print(cbc_cleaned.columns.tolist())

print("\nCMP columns:")
print(cmp_cleaned.columns.tolist())

CBC shape: (553, 10)
CMP shape: (28, 52)

CBC columns:
['ANALYTE', 'VALUE', 'RANGE_MIN', 'RANGE_MAX', 'UNITS', 'TEST_TYPE', 'SUBJECT_ID', 'SEX', 'TEST_DATE', 'raw_analyte']

CMP columns:
['Sample Name', 'albumin_value_gram_per_deciliter', 'albumin_range_min_gram_per_deciliter', 'albumin_range_max_gram_per_deciliter', 'albumin_to_globulin_ratio_value', 'albumin_to_globulin_ratio_range_min', 'albumin_to_globulin_ratio_range_max', 'alkaline_phosphatase_value_units_per_liter', 'alkaline_phosphatase_range_min_units_per_liter', 'alkaline_phosphatase_range_max_units_per_liter', 'alt_value_units_per_liter', 'alt_range_min_units_per_liter', 'alt_range_max_units_per_liter', 'ast_value_units_per_liter', 'ast_range_min_units_per_liter', 'ast_range_max_units_per_liter', 'total_bilirubin_value_milligram_per_deciliter', 'total_bilirubin_range_min_milligram_per_deciliter', 'total_bilirubin_range_max_milligram_per_deciliter', 'calcium_value_milligram_per_deciliter', 'calcium_range_min_milligram_per_dec

In [7]:
# Select and standardize CBC columns

cbc = cbc_cleaned[
    ["SUBJECT_ID", "TEST_DATE", "ANALYTE",
     "VALUE", "UNITS", "RANGE_MIN", "RANGE_MAX"]
].copy()

cbc["PANEL"] = "CBC"

cbc = cbc[
    ["SUBJECT_ID", "TEST_DATE", "PANEL", "ANALYTE",
     "VALUE", "UNITS", "RANGE_MIN", "RANGE_MAX"]
]

print("CBC shape:", cbc.shape)
display(cbc.head())

CBC shape: (553, 8)


,SUBJECT_ID,TEST_DATE,PANEL,ANALYTE,VALUE,UNITS,RANGE_MIN,RANGE_MAX
0,C001,L-92,CBC,WHITE BLOOD CELL COUNT,5.40,Thousand/uL,3.8,10.8
1,C001,L-92,CBC,RED BLOOD CELL COUNT,4.58,Million/uL,4.2,5.8
2,C001,L-92,CBC,HEMOGLOBIN,14.10,g/dL,13.2,17.1
3,C001,L-92,CBC,HEMATOCRIT,42.50,%,38.5,50.0
4,C001,L-92,CBC,MCV,92.80,fL,80.0,100.0


In [8]:
# Identify CMP value columns

value_cols = [
    col for col in cmp_cleaned.columns
    if "_value_" in col or col.endswith("_value")
]

print("Number of CMP biomarkers:", len(value_cols))
print(value_cols)

Number of CMP biomarkers: 16
['albumin_value_gram_per_deciliter', 'albumin_to_globulin_ratio_value', 'alkaline_phosphatase_value_units_per_liter', 'alt_value_units_per_liter', 'ast_value_units_per_liter', 'total_bilirubin_value_milligram_per_deciliter', 'calcium_value_milligram_per_deciliter', 'carbon_dioxide_value_millimol_per_liter', 'chloride_value_millimol_per_liter', 'creatinine_value_milligram_per_deciliter', 'globulin_value_gram_per_deciliter', 'glucose_value_milligram_per_deciliter', 'potassium_value_millimol_per_liter', 'total_protein_value_gram_per_deciliter', 'sodium_value_millimol_per_liter', 'urea_nitrogen_bun_value_milligram_per_deciliter']


In [9]:
# Convert CMP from wide to long format

id_cols = ["Sample Name", "SUBJECT_ID", "TEST_DATE"]

cmp_long = cmp_cleaned.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name="ANALYTE",
    value_name="VALUE"
)

# Clean the biomarker names
cmp_long["ANALYTE"] = (
    cmp_long["ANALYTE"]
    .str.replace("_value_", "_", regex=False)
    .str.replace("_value", "", regex=False)
)

# Add panel name
cmp_long["PANEL"] = "CMP"

print("CMP long shape:", cmp_long.shape)
display(cmp_long.head())

CMP long shape: (448, 6)


,Sample Name,SUBJECT_ID,TEST_DATE,ANALYTE,VALUE,PANEL
0,C001_serum_L-3,C001,L-3,albumin_gram_per_deciliter,4.9,CMP
1,C001_serum_L-44,C001,L-44,albumin_gram_per_deciliter,5.0,CMP
2,C001_serum_L-92,C001,L-92,albumin_gram_per_deciliter,5.0,CMP
3,C001_serum_R+1,C001,R+1,albumin_gram_per_deciliter,4.9,CMP
4,C001_serum_R+194,C001,R+194,albumin_gram_per_deciliter,4.6,CMP


In [10]:
# Create a mapping for CMP biomarkers

mapping = []

for col in value_cols:
    if "_value_" in col:
        name, unit = col.split("_value_", 1)
    else:
        name = col.removesuffix("_value")
        unit = ""

    mapping.append({
        "value_col": col,
        "analyte": col.replace("_value_", "_", 1).removesuffix("_value"),
        "min_col": f"{name}_range_min_{unit}" if unit else f"{name}_range_min",
        "max_col": f"{name}_range_max_{unit}" if unit else f"{name}_range_max",
        "units": unit
    })

mapping = pd.DataFrame(mapping)

display(mapping)

,value_col,analyte,min_col,max_col,units
0,albumin_value_gram_per_deciliter,albumin_gram_per_deciliter,albumin_range_min_gram_per_deciliter,albumin_range_max_gram_per_deciliter,gram_per_deciliter
1,albumin_to_globulin_ratio_value,albumin_to_globulin_ratio,albumin_to_globulin_ratio_range_min,albumin_to_globulin_ratio_range_max,
2,alkaline_phosphatase_value_units_per_liter,alkaline_phosphatase_units_per_liter,alkaline_phosphatase_range_min_units_per_liter,alkaline_phosphatase_range_max_units_per_liter,units_per_liter
3,alt_value_units_per_liter,alt_units_per_liter,alt_range_min_units_per_liter,alt_range_max_units_per_liter,units_per_liter
4,ast_value_units_per_liter,ast_units_per_liter,ast_range_min_units_per_liter,ast_range_max_units_per_liter,units_per_liter
5,total_bilirubin_value_milligram_per_deciliter,total_bilirubin_milligram_per_deciliter,total_bilirubin_range_min_milligram_per_deciliter,total_bilirubin_range_max_milligram_per_deciliter,milligram_per_deciliter
6,calcium_value_milligram_per_deciliter,calcium_milligram_per_deciliter,calcium_range_min_milligram_per_deciliter,calcium_range_max_milligram_per_deciliter,milligram_per_deciliter
7,carbon_dioxide_value_millimol_per_liter,carbon_dioxide_millimol_per_liter,carbon_dioxide_range_min_millimol_per_liter,carbon_dioxide_range_max_millimol_per_liter,millimol_per_liter
8,chloride_value_millimol_per_liter,chloride_millimol_per_liter,chloride_range_min_millimol_per_liter,chloride_range_max_millimol_per_liter,millimol_per_liter
9,creatinine_value_milligram_per_deciliter,creatinine_milligram_per_deciliter,creatinine_range_min_milligram_per_deciliter,creatinine_range_max_milligram_per_deciliter,milligram_per_deciliter


In [11]:
# Check that every mapped column exists

mapping["min_exists"] = mapping["min_col"].isin(cmp_cleaned.columns)
mapping["max_exists"] = mapping["max_col"].isin(cmp_cleaned.columns)

display(mapping[
    ["analyte", "min_col", "max_col", "min_exists", "max_exists"]
])

print("Missing minimum columns:", (~mapping["min_exists"]).sum())
print("Missing maximum columns:", (~mapping["max_exists"]).sum())

,analyte,min_col,max_col,min_exists,max_exists
0,albumin_gram_per_deciliter,albumin_range_min_gram_per_deciliter,albumin_range_max_gram_per_deciliter,True,True
1,albumin_to_globulin_ratio,albumin_to_globulin_ratio_range_min,albumin_to_globulin_ratio_range_max,True,True
2,alkaline_phosphatase_units_per_liter,alkaline_phosphatase_range_min_units_per_liter,alkaline_phosphatase_range_max_units_per_liter,True,True
3,alt_units_per_liter,alt_range_min_units_per_liter,alt_range_max_units_per_liter,True,True
4,ast_units_per_liter,ast_range_min_units_per_liter,ast_range_max_units_per_liter,True,True
5,total_bilirubin_milligram_per_deciliter,total_bilirubin_range_min_milligram_per_deciliter,total_bilirubin_range_max_milligram_per_deciliter,True,True
6,calcium_milligram_per_deciliter,calcium_range_min_milligram_per_deciliter,calcium_range_max_milligram_per_deciliter,True,True
7,carbon_dioxide_millimol_per_liter,carbon_dioxide_range_min_millimol_per_liter,carbon_dioxide_range_max_millimol_per_liter,True,True
8,chloride_millimol_per_liter,chloride_range_min_millimol_per_liter,chloride_range_max_millimol_per_liter,True,True
9,creatinine_milligram_per_deciliter,creatinine_range_min_milligram_per_deciliter,creatinine_range_max_milligram_per_deciliter,True,True


Missing minimum columns: 0
Missing maximum columns: 0


In [12]:
# Build the CMP dataset in a common long format

cmp_rows = []

for _, row in mapping.iterrows():
    part = cmp_cleaned[
        ["Sample Name", "SUBJECT_ID", "TEST_DATE",
         row["value_col"], row["min_col"], row["max_col"]]
    ].copy()

    part.columns = [
        "Sample Name", "SUBJECT_ID", "TEST_DATE",
        "VALUE", "RANGE_MIN", "RANGE_MAX"
    ]

    part["ANALYTE"] = row["analyte"]
    part["UNITS"] = row["units"] if row["units"] else pd.NA
    part["PANEL"] = "CMP"

    cmp_rows.append(part)

cmp = pd.concat(cmp_rows, ignore_index=True)

cmp = cmp[
    ["SUBJECT_ID", "TEST_DATE", "PANEL", "ANALYTE",
     "VALUE", "UNITS", "RANGE_MIN", "RANGE_MAX"]
]

print("CMP shape:", cmp.shape)
display(cmp.head())

CMP shape: (448, 8)


,SUBJECT_ID,TEST_DATE,PANEL,ANALYTE,VALUE,UNITS,RANGE_MIN,RANGE_MAX
0,C001,L-3,CMP,albumin_gram_per_deciliter,4.9,gram_per_deciliter,3.6,5.1
1,C001,L-44,CMP,albumin_gram_per_deciliter,5.0,gram_per_deciliter,3.6,5.1
2,C001,L-92,CMP,albumin_gram_per_deciliter,5.0,gram_per_deciliter,3.6,5.1
3,C001,R+1,CMP,albumin_gram_per_deciliter,4.9,gram_per_deciliter,3.6,5.1
4,C001,R+194,CMP,albumin_gram_per_deciliter,4.6,gram_per_deciliter,3.6,5.1


In [13]:
# Combine CBC and CMP

data = pd.concat([cbc, cmp], ignore_index=True)

# Set consistent column order
cols = [
    "SUBJECT_ID", "TEST_DATE", "PANEL", "ANALYTE",
    "VALUE", "UNITS", "RANGE_MIN", "RANGE_MAX"
]

data = data[cols]

print("Combined shape:", data.shape)
print("\nRows by panel:")
print(data["PANEL"].value_counts())

display(data.head())

Combined shape: (1001, 8)

Rows by panel:
PANEL
CBC    553
CMP    448
Name: count, dtype: int64


,SUBJECT_ID,TEST_DATE,PANEL,ANALYTE,VALUE,UNITS,RANGE_MIN,RANGE_MAX
0,C001,L-92,CBC,WHITE BLOOD CELL COUNT,5.40,Thousand/uL,3.8,10.8
1,C001,L-92,CBC,RED BLOOD CELL COUNT,4.58,Million/uL,4.2,5.8
2,C001,L-92,CBC,HEMOGLOBIN,14.10,g/dL,13.2,17.1
3,C001,L-92,CBC,HEMATOCRIT,42.50,%,38.5,50.0
4,C001,L-92,CBC,MCV,92.80,fL,80.0,100.0


In [14]:
# Check dataset quality

print("Unique participants:", data["SUBJECT_ID"].nunique())
print("Timepoints:", sorted(data["TEST_DATE"].unique()))

print("\nMissing values:")
display(data.isna().sum().to_frame("missing_count"))

print("\nDuplicate measurements:")
print(
    data.duplicated(
        subset=["SUBJECT_ID", "TEST_DATE", "PANEL", "ANALYTE"]
    ).sum()
)

Unique participants: 4
Timepoints: ['L-3', 'L-44', 'L-92', 'R+1', 'R+194', 'R+45', 'R+82']

Missing values:


,missing_count
SUBJECT_ID,0
TEST_DATE,0
PANEL,0
ANALYTE,0
VALUE,1
UNITS,28
RANGE_MIN,0
RANGE_MAX,0



Duplicate measurements:
0


In [15]:
# Select preflight measurements

pre = data[data["TEST_DATE"].isin(["L-92", "L-44", "L-3"])].copy()

# Calculate personal baselines
base = (
    pre.groupby(["SUBJECT_ID", "PANEL", "ANALYTE"])["VALUE"]
    .agg(
        base_count="count",
        base_mean="mean",
        base_std="std"
    )
    .reset_index()
)

print("Baseline table shape:", base.shape)
display(base.head(10))

Baseline table shape: (143, 6)


,SUBJECT_ID,PANEL,ANALYTE,base_count,base_mean,base_std
0,C001,CBC,ABSOLUTE BASOPHILS,3,56.333333,11.846237
1,C001,CBC,ABSOLUTE EOSINOPHILS,3,208.000000,49.122296
2,C001,CBC,ABSOLUTE LYMPHOCYTES,3,2255.666667,404.199621
3,C001,CBC,ABSOLUTE MONOCYTES,3,475.000000,81.166496
4,C001,CBC,ABSOLUTE NEUTROPHILS,3,2972.333333,610.904521
5,C001,CBC,BASOPHILS,3,0.966667,0.305505
6,C001,CBC,EOSINOPHILS,3,3.600000,1.252996
7,C001,CBC,HEMATOCRIT,3,44.033333,1.361372
8,C001,CBC,HEMOGLOBIN,3,14.566667,0.416333
9,C001,CBC,LYMPHOCYTES,3,37.800000,3.858756


In [16]:
print("Baseline measurement counts:")
display(base["base_count"].value_counts().sort_index())

print("\nMissing baseline standard deviations:")
print(base["base_std"].isna().sum())

print("\nZero baseline standard deviations:")
print((base["base_std"] == 0).sum())

Baseline measurement counts:


,count
base_count,
2,2
3,141



Missing baseline standard deviations:
0

Zero baseline standard deviations:
0


In [17]:
# Select postflight measurements

post = data[
    data["TEST_DATE"].isin(["R+1", "R+45", "R+82", "R+194"])
].copy()

# Join each measurement to its personal baseline
scores = post.merge(
    base,
    on=["SUBJECT_ID", "PANEL", "ANALYTE"],
    how="left"
)

# Calculate baseline-relative z-scores
scores["z_score"] = (
    (scores["VALUE"] - scores["base_mean"])
    / scores["base_std"]
)

# Flag unusually large deviations
scores["flag"] = scores["z_score"].abs() > 2

# Missing measurements should not be classified as normal
scores.loc[scores["VALUE"].isna(), "flag"] = pd.NA
scores["flag"] = scores["flag"].astype("boolean")

print("Postflight shape:", post.shape)
print("Scored shape:", scores.shape)

display(
    scores[
        ["SUBJECT_ID", "TEST_DATE", "PANEL", "ANALYTE",
         "VALUE", "base_mean", "base_std", "z_score", "flag"]
    ].head(10)
)

Postflight shape: (574, 8)
Scored shape: (574, 13)


/tmp/ipykernel_1189/217191138.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  scores.loc[scores["VALUE"].isna(), "flag"] = pd.NA


,SUBJECT_ID,TEST_DATE,PANEL,ANALYTE,VALUE,base_mean,base_std,z_score,flag
0,C001,R+1,CBC,WHITE BLOOD CELL COUNT,5.00,5.966667,0.896289,-1.078522,False
1,C001,R+1,CBC,RED BLOOD CELL COUNT,4.84,4.706667,0.135769,0.982057,False
2,C001,R+1,CBC,HEMOGLOBIN,14.70,14.566667,0.416333,0.320256,False
3,C001,R+1,CBC,HEMATOCRIT,43.60,44.033333,1.361372,-0.318306,False
4,C001,R+1,CBC,MCV,90.10,93.566667,1.159023,-2.991026,True
5,C001,R+1,CBC,MCH,30.40,30.933333,0.321455,-1.659123,False
6,C001,R+1,CBC,MCHC,33.70,33.066667,0.115470,5.484828,True
7,C001,R+1,CBC,RDW,12.50,12.733333,0.230940,-1.010363,False
8,C001,R+1,CBC,PLATELET COUNT,227.00,245.333333,11.150486,-1.644174,False
9,C001,R+1,CBC,MPV,10.80,10.600000,0.300000,0.666667,False


In [18]:
print("Flag counts:")
display(scores["flag"].value_counts(dropna=False))

print("\nMissing z-scores:", scores["z_score"].isna().sum())

print("\nFlags by participant:")
display(
    scores.groupby("SUBJECT_ID")["flag"]
    .agg(
        total="count",
        flagged=lambda x: (x == True).sum()
    )
)

Flag counts:


,count
flag,
False,441
True,132
<NA>,1



Missing z-scores: 4

Flags by participant:


,total,flagged
SUBJECT_ID,,
C001,143,19
C002,142,31
C003,144,40
C004,144,42


In [19]:
# Calculate z-scores
scores["z_score"] = (
    (scores["VALUE"] - scores["base_mean"])
    / scores["base_std"]
)

# Use nullable Boolean values
scores["flag"] = (
    scores["z_score"].abs() > 2
).astype("boolean")

# Missing z-scores should have unknown flags
scores.loc[scores["z_score"].isna(), "flag"] = pd.NA

print(scores["flag"].value_counts(dropna=False))

flag
False    438
True     132
<NA>       4
Name: count, dtype: Int64


In [20]:
# Inspect rows that could not be scored

missing_scores = scores[scores["z_score"].isna()]

display(
    missing_scores[
        [
            "SUBJECT_ID", "TEST_DATE", "PANEL", "ANALYTE",
            "VALUE", "base_count", "base_mean", "base_std"
        ]
    ]
)

,SUBJECT_ID,TEST_DATE,PANEL,ANALYTE,VALUE,base_count,base_mean,base_std
105,C002,R+45,CBC,MCHC,32.9,NaN,NaN,NaN
125,C002,R+82,CBC,MCHC,33.1,NaN,NaN,NaN
144,C002,R+194,CBC,MCHC,33.2,NaN,NaN,NaN
400,C001,R+45,CMP,total_bilirubin_milligram_per_deciliter,NaN,3.0,0.866667,0.152753


In [21]:
# Review the largest absolute z-scores

display(
    scores.loc[scores["z_score"].abs().nlargest(15).index, [
        "SUBJECT_ID", "TEST_DATE", "PANEL", "ANALYTE",
        "VALUE", "base_count", "base_mean", "base_std", "z_score"
    ]].sort_values("z_score", key=lambda x: x.abs(), ascending=False)
)

,SUBJECT_ID,TEST_DATE,PANEL,ANALYTE,VALUE,base_count,base_mean,base_std,z_score
236,C004,R+194,CBC,EOSINOPHILS,4.5,3.0,2.633333,0.057735,32.331615
196,C004,R+45,CBC,EOSINOPHILS,3.8,3.0,2.633333,0.057735,20.207259
176,C004,R+1,CBC,EOSINOPHILS,3.7,3.0,2.633333,0.057735,18.475209
216,C004,R+82,CBC,EOSINOPHILS,3.4,3.0,2.633333,0.057735,13.279056
381,C004,R+82,CMP,alt_units_per_liter,23.0,3.0,36.333333,1.154701,-11.547005
374,C003,R+1,CMP,alt_units_per_liter,19.0,3.0,12.666667,0.577350,10.969655
298,C003,R+1,CBC,WHITE BLOOD CELL COUNT,7.0,3.0,9.566667,0.251661,-10.198899
113,C002,R+45,CBC,ABSOLUTE BASOPHILS,51.0,3.0,31.333333,2.081666,9.447561
46,C001,R+82,CBC,MCHC,34.1,3.0,33.066667,0.115470,8.948929
278,C003,R+194,CBC,WHITE BLOOD CELL COUNT,7.4,3.0,9.566667,0.251661,-8.609460


In [22]:
# Classify measurements using the supplied reference limits

conditions = [
    scores["VALUE"].isna()
    | scores["RANGE_MIN"].isna()
    | scores["RANGE_MAX"].isna(),

    scores["VALUE"] < scores["RANGE_MIN"],

    scores["VALUE"] > scores["RANGE_MAX"]
]

choices = ["missing", "below", "above"]

scores["range_status"] = np.select(
    conditions,
    choices,
    default="within"
)

print("Reference-range status:")
display(scores["range_status"].value_counts(dropna=False))

Reference-range status:


,count
range_status,
within,555
above,10
below,8
missing,1


In [23]:
print("Baseline-relative flags:")
display(scores["flag"].value_counts(dropna=False))

print("\nReference-range status by panel:")
display(
    pd.crosstab(
        scores["PANEL"],
        scores["range_status"],
        margins=True
    )
)

Baseline-relative flags:


,count
flag,
False,438
True,132
<NA>,4



Reference-range status by panel:


range_status,above,below,missing,within,All
PANEL,,,,,
CBC,3,2,0,313,318
CMP,7,6,1,242,256
All,10,8,1,555,574


In [24]:
# Add a readable baseline status

scores["baseline_status"] = "within_baseline_threshold"

scores.loc[
    scores["flag"] == True,
    "baseline_status"
] = "baseline_deviation"

scores.loc[
    scores["flag"].isna(),
    "baseline_status"
] = "unscored"

# Select dashboard columns
dashboard = scores[
    [
        "SUBJECT_ID",
        "TEST_DATE",
        "PANEL",
        "ANALYTE",
        "VALUE",
        "UNITS",
        "RANGE_MIN",
        "RANGE_MAX",
        "range_status",
        "base_mean",
        "base_std",
        "base_count",
        "z_score",
        "baseline_status"
    ]
].copy()

print("Dashboard dataset shape:", dashboard.shape)
display(dashboard.head(10))

Dashboard dataset shape: (574, 14)


,SUBJECT_ID,TEST_DATE,PANEL,ANALYTE,VALUE,UNITS,RANGE_MIN,RANGE_MAX,range_status,base_mean,base_std,base_count,z_score,baseline_status
0,C001,R+1,CBC,WHITE BLOOD CELL COUNT,5.00,Thousand/uL,3.8,10.8,within,5.966667,0.896289,3.0,-1.078522,within_baseline_threshold
1,C001,R+1,CBC,RED BLOOD CELL COUNT,4.84,Million/uL,4.2,5.8,within,4.706667,0.135769,3.0,0.982057,within_baseline_threshold
2,C001,R+1,CBC,HEMOGLOBIN,14.70,g/dL,13.2,17.1,within,14.566667,0.416333,3.0,0.320256,within_baseline_threshold
3,C001,R+1,CBC,HEMATOCRIT,43.60,%,38.5,50.0,within,44.033333,1.361372,3.0,-0.318306,within_baseline_threshold
4,C001,R+1,CBC,MCV,90.10,fL,80.0,100.0,within,93.566667,1.159023,3.0,-2.991026,baseline_deviation
5,C001,R+1,CBC,MCH,30.40,pg,27.0,33.0,within,30.933333,0.321455,3.0,-1.659123,within_baseline_threshold
6,C001,R+1,CBC,MCHC,33.70,g/dL,32.0,36.0,within,33.066667,0.115470,3.0,5.484828,baseline_deviation
7,C001,R+1,CBC,RDW,12.50,%,11.0,15.0,within,12.733333,0.230940,3.0,-1.010363,within_baseline_threshold
8,C001,R+1,CBC,PLATELET COUNT,227.00,Thousand/uL,140.0,400.0,within,245.333333,11.150486,3.0,-1.644174,within_baseline_threshold
9,C001,R+1,CBC,MPV,10.80,fL,7.5,12.5,within,10.600000,0.300000,3.0,0.666667,within_baseline_threshold


In [25]:
print("Baseline status:")
display(dashboard["baseline_status"].value_counts(dropna=False))

print("\nMissing values by column:")
display(dashboard.isna().sum().to_frame("missing_count"))

print("\nDuplicate measurements:")
print(
    dashboard.duplicated(
        subset=["SUBJECT_ID", "TEST_DATE", "PANEL", "ANALYTE"]
    ).sum()
)

Baseline status:


,count
baseline_status,
within_baseline_threshold,438
baseline_deviation,132
unscored,4



Missing values by column:


,missing_count
SUBJECT_ID,0
TEST_DATE,0
PANEL,0
ANALYTE,0
VALUE,1
UNITS,16
RANGE_MIN,0
RANGE_MAX,0
range_status,0
base_mean,3



Duplicate measurements:
0


In [26]:
dashboard.to_csv("astrohealth_dashboard.csv", index=False)

print("Dashboard dataset exported successfully!")

Dashboard dataset exported successfully!


In [27]:
from google.colab import files

files.download("astrohealth_dashboard.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
thresholds = [2, 2.5, 3]

for threshold in thresholds:
    column = f"flag_{str(threshold).replace('.', '_')}"

    scores[column] = (
        scores["z_score"].abs() > threshold
    ).astype("boolean")

    scores.loc[scores["z_score"].isna(), column] = pd.NA

    print(
        f"Threshold {threshold}: "
        f"{scores[column].sum()} flagged measurements"
    )

Threshold 2: 132 flagged measurements
Threshold 2.5: 93 flagged measurements
Threshold 3: 79 flagged measurements


In [29]:
# Examine the baseline standard deviations
baseline_check = (
    base.sort_values("base_std")
    [["SUBJECT_ID", "PANEL", "ANALYTE",
      "base_count", "base_mean", "base_std"]]
)

display(baseline_check.head(20))

,SUBJECT_ID,PANEL,ANALYTE,base_count,base_mean,base_std
135,C004,CMP,creatinine_milligram_per_deciliter,3,1.116667,0.032146
99,C003,CMP,creatinine_milligram_per_deciliter,3,1.036667,0.040415
20,C001,CMP,albumin_gram_per_deciliter,3,4.966667,0.057735
76,C003,CBC,BASOPHILS,3,0.466667,0.057735
113,C004,CBC,EOSINOPHILS,3,2.633333,0.057735
56,C002,CMP,albumin_to_globulin_ratio,3,1.566667,0.057735
141,C004,CMP,total_protein_gram_per_deciliter,3,7.333333,0.057735
63,C002,CMP,creatinine_milligram_per_deciliter,3,0.890000,0.065574
89,C003,CBC,RED BLOOD CELL COUNT,3,4.526667,0.083267
125,C004,CBC,RED BLOOD CELL COUNT,3,5.483333,0.085049


In [30]:
flagged = scores[scores["flag"] == True].copy()

display(
    flagged.sort_values("base_std")[
        ["SUBJECT_ID", "TEST_DATE", "PANEL", "ANALYTE",
         "VALUE", "base_mean", "base_std",
         "base_count", "z_score", "range_status"]
    ].head(20)
)

,SUBJECT_ID,TEST_DATE,PANEL,ANALYTE,VALUE,base_mean,base_std,base_count,z_score,range_status
475,C004,R+194,CMP,creatinine_milligram_per_deciliter,1.31,1.116667,0.032146,3.0,6.014320,within
470,C003,R+1,CMP,creatinine_milligram_per_deciliter,0.89,1.036667,0.040415,3.0,-3.629059,within
472,C003,R+45,CMP,creatinine_milligram_per_deciliter,0.94,1.036667,0.040415,3.0,-2.391880,within
473,C003,R+82,CMP,creatinine_milligram_per_deciliter,0.89,1.036667,0.040415,3.0,-3.629059,within
319,C001,R+194,CMP,albumin_gram_per_deciliter,4.60,4.966667,0.057735,3.0,-6.350853,within
321,C001,R+82,CMP,albumin_gram_per_deciliter,5.10,4.966667,0.057735,3.0,2.309401,within
257,C003,R+45,CBC,BASOPHILS,0.80,0.466667,0.057735,3.0,5.773503,within
277,C003,R+82,CBC,BASOPHILS,0.80,0.466667,0.057735,3.0,5.773503,within
297,C003,R+194,CBC,BASOPHILS,0.80,0.466667,0.057735,3.0,5.773503,within
216,C004,R+82,CBC,EOSINOPHILS,3.40,2.633333,0.057735,3.0,13.279056,within


In [31]:
# Focus on measurements that received a baseline flag
flagged = scores[scores["flag"] == True].copy()

print("Total baseline flags:", len(flagged))

print(
    "Flags within reference range:",
    (flagged["range_status"] == "within").sum()
)

print(
    "Flags outside reference range:",
    flagged["range_status"].isin(["above", "below"]).sum()
)

print(
    "Flags with baseline SD below 0.1:",
    (flagged["base_std"] < 0.1).sum()
)

print(
    "Flags with baseline SD below 0.1 and within range:",
    (
        (flagged["base_std"] < 0.1)
        & (flagged["range_status"] == "within")
    ).sum()
)

Total baseline flags: 132
Flags within reference range: 120
Flags outside reference range: 12
Flags with baseline SD below 0.1: 24
Flags with baseline SD below 0.1 and within range: 24


In [32]:
# Find flagged measurements with small baseline SD
small_sd = flagged[
    (flagged["base_std"] < 0.1)
    & (flagged["range_status"] == "within")
].copy()

# Retrieve the original preflight measurements
baseline_values = pre.merge(
    small_sd[["SUBJECT_ID", "PANEL", "ANALYTE"]].drop_duplicates(),
    on=["SUBJECT_ID", "PANEL", "ANALYTE"],
    how="inner"
)

baseline_values = baseline_values.sort_values(
    ["SUBJECT_ID", "PANEL", "ANALYTE", "TEST_DATE"]
)

display(
    baseline_values[
        ["SUBJECT_ID", "PANEL", "ANALYTE", "TEST_DATE", "VALUE"]
    ]
)

,SUBJECT_ID,PANEL,ANALYTE,TEST_DATE,VALUE
12,C001,CMP,albumin_gram_per_deciliter,L-3,4.90
13,C001,CMP,albumin_gram_per_deciliter,L-44,5.00
14,C001,CMP,albumin_gram_per_deciliter,L-92,5.00
2,C002,CBC,BASOPHILS,L-3,0.40
1,C002,CBC,BASOPHILS,L-44,0.60
0,C002,CBC,BASOPHILS,L-92,0.50
15,C002,CMP,albumin_to_globulin_ratio,L-3,1.60
16,C002,CMP,albumin_to_globulin_ratio,L-44,1.50
17,C002,CMP,albumin_to_globulin_ratio,L-92,1.60
11,C003,CBC,BASOPHILS,L-3,0.50


In [33]:
# Count observations available for each participant and biomarker
timepoint_counts = (
    data.groupby(["SUBJECT_ID", "PANEL", "ANALYTE"])
    .agg(
        observations=("VALUE", "count"),
        timepoints=("TEST_DATE", "nunique")
    )
    .reset_index()
)

print("Observation count distribution:")
display(timepoint_counts["observations"].value_counts().sort_index())

print("\nBiomarkers with fewer than 7 observations:")
display(
    timepoint_counts[timepoint_counts["observations"] < 7]
    .sort_values(["SUBJECT_ID", "PANEL", "ANALYTE"])
)

Observation count distribution:


,count
observations,
3,1
6,4
7,139



Biomarkers with fewer than 7 observations:


,SUBJECT_ID,PANEL,ANALYTE,observations,timepoints
33,C001,CMP,total_bilirubin_milligram_per_deciliter,6,7
47,C002,CBC,MCHC,3,3
48,C002,CBC,MCV,6,6
82,C003,CBC,MCH,6,6
118,C004,CBC,MCH,6,6


In [34]:
expected_timepoints = [
    "L-92", "L-44", "L-3",
    "R+1", "R+45", "R+82", "R+194"
]

timepoint_check = (
    data.groupby(["SUBJECT_ID", "PANEL", "ANALYTE"])["TEST_DATE"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="available_timepoints")
)

timepoint_check["missing_timepoints"] = timepoint_check[
    "available_timepoints"
].apply(lambda x: sorted(set(expected_timepoints) - set(x)))

display(
    timepoint_check[
        timepoint_check["missing_timepoints"].apply(bool)
    ]
)

,SUBJECT_ID,PANEL,ANALYTE,available_timepoints,missing_timepoints
47,C002,CBC,MCHC,"[R+194, R+45, R+82]","[L-3, L-44, L-92, R+1]"
48,C002,CBC,MCV,"[L-3, L-44, L-92, R+1, R+45, R+82]",[R+194]
82,C003,CBC,MCH,"[L-3, L-44, R+1, R+194, R+45, R+82]",[L-92]
118,C004,CBC,MCH,"[L-3, L-44, R+1, R+194, R+45, R+82]",[L-92]


In [35]:
key = ["SUBJECT_ID", "PANEL", "ANALYTE", "TEST_DATE"]

duplicates = data[data.duplicated(key, keep=False)]

print("Rows involved in duplicate measurements:", len(duplicates))
display(duplicates.sort_values(key))

Rows involved in duplicate measurements: 0


,SUBJECT_ID,TEST_DATE,PANEL,ANALYTE,VALUE,UNITS,RANGE_MIN,RANGE_MAX


In [36]:
baseline_timepoints = ["L-92", "L-44", "L-3"]

baseline = data[data["TEST_DATE"].isin(baseline_timepoints)]

baseline_counts = (
    baseline.groupby(["SUBJECT_ID", "PANEL", "ANALYTE"])
    .agg(
        baseline_observations=("VALUE", "count"),
        baseline_timepoints=("TEST_DATE", "nunique")
    )
    .reset_index()
)

display(
    baseline_counts[
        (baseline_counts["baseline_observations"] < 3)
        | (baseline_counts["baseline_timepoints"] < 3)
    ]
)

,SUBJECT_ID,PANEL,ANALYTE,baseline_observations,baseline_timepoints
81,C003,CBC,MCH,2,2
117,C004,CBC,MCH,2,2


In [37]:
range_check = (
    data.groupby(["SUBJECT_ID", "PANEL", "ANALYTE"])
    .agg(
        min_range_values=("RANGE_MIN", "nunique"),
        max_range_values=("RANGE_MAX", "nunique")
    )
    .reset_index()
)

inconsistent_ranges = range_check[
    (range_check["min_range_values"] > 1)
    | (range_check["max_range_values"] > 1)
]

display(inconsistent_ranges)

,SUBJECT_ID,PANEL,ANALYTE,min_range_values,max_range_values
79,C003,CBC,HEMATOCRIT,2,2
80,C003,CBC,HEMOGLOBIN,2,2
90,C003,CBC,RED BLOOD CELL COUNT,2,2
94,C003,CMP,alkaline_phosphatase_units_per_liter,2,2
95,C003,CMP,alt_units_per_liter,2,2
96,C003,CMP,ast_units_per_liter,1,2
97,C003,CMP,calcium_milligram_per_deciliter,1,2
100,C003,CMP,creatinine_milligram_per_deciliter,2,2


In [38]:
flagged = inconsistent_ranges[
    ["SUBJECT_ID", "PANEL", "ANALYTE"]
]

range_details = data.merge(
    flagged,
    on=["SUBJECT_ID", "PANEL", "ANALYTE"],
    how="inner"
)

display(
    range_details[
        [
            "SUBJECT_ID", "PANEL", "ANALYTE",
            "TEST_DATE", "VALUE", "UNITS",
            "RANGE_MIN", "RANGE_MAX"
        ]
    ].sort_values(
        ["PANEL", "ANALYTE", "TEST_DATE"]
    )
)

,SUBJECT_ID,PANEL,ANALYTE,TEST_DATE,VALUE,UNITS,RANGE_MIN,RANGE_MAX
8,C003,CBC,HEMATOCRIT,L-3,41.60,%,38.5,50.00
5,C003,CBC,HEMATOCRIT,L-44,42.60,%,38.5,50.00
2,C003,CBC,HEMATOCRIT,L-92,40.70,%,38.5,50.00
20,C003,CBC,HEMATOCRIT,R+1,41.40,%,35.0,45.00
17,C003,CBC,HEMATOCRIT,R+194,40.40,%,35.0,45.00
11,C003,CBC,HEMATOCRIT,R+45,44.30,%,35.0,45.00
14,C003,CBC,HEMATOCRIT,R+82,44.60,%,35.0,45.00
7,C003,CBC,HEMOGLOBIN,L-3,13.30,g/dL,13.2,17.10
4,C003,CBC,HEMOGLOBIN,L-44,14.40,g/dL,13.2,17.10
1,C003,CBC,HEMOGLOBIN,L-92,13.20,g/dL,13.2,17.10
